# 📱 Smartphone Data Cleaning & Preprocessing

This notebook documents a practical **data cleaning and preprocessing workflow** on a smartphone dataset.

The objective is to assess the raw data, identify data-quality problems, clean inconsistent values, and transform complex columns into structured variables that are easier to analyze.

### Data Quality Dimensions Covered

- **Completeness** — missing values in important fields
- **Validity** — incorrect data types, formats, or values
- **Accuracy** — values that are placed in the wrong column or do not represent the intended information
- **Consistency** — different representations of the same information
- **Uniqueness** — duplicate records
- **Tidiness / Messy Data** — multiple variables stored together or values shifted into the wrong fields

> **Source note:** This is my personal practice and implementation of data-cleaning techniques learned through the CampusX DSMP 2 course. The explanations in this notebook describe the analysis and transformations performed here.


In [390]:
import numpy as np
import pandas as pd

## 📥 Loading the Dataset

The smartphone CSV file is loaded into a Pandas DataFrame.

The raw dataset is kept separate from the working DataFrame so that the original data can be preserved while cleaning operations are performed.


In [391]:
df=pd.read_csv('/content/smartphones - smartphones (1).csv')

In [392]:
df.head()

,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,OnePlus 11 5G,"₹54,999",89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,OnePlus Nord CE 2 Lite 5G,"₹19,989",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,Samsung Galaxy A14 5G,"₹16,499",75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,Motorola Moto G62 5G,"₹14,999",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,Realme 10 Pro Plus,"₹24,999",82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13


## 🔎 Initial Data Assessment — Programmatic Assessment

Before changing the data, we inspect it programmatically.

The assessment focuses on:

- missing values
- incorrect data types
- unusual values
- duplicate records
- inconsistent column contents

The principle is:

> **Assess first → identify the issue → decide how to clean it → apply the transformation.**


Programmatic Assesment

### 1. Checking Structure, Missing Values and Data Types

`info()` gives an overview of the DataFrame, including the number of rows, columns, non-null values, and data types.

The initial assessment identifies missing values in columns such as `rating`, `card`, `camera`, and `os`, while `price` and `rating` require attention regarding their data types.

**Data Quality Dimensions:** Completeness and Validity

**Issue Type:** Dirty data


In [393]:
#missing values in rating,card,camera,os
#datatype of price and rating is not correct
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   model      1020 non-null   object 
 1   price      1020 non-null   object 
 2   rating     879 non-null    float64
 3   sim        1020 non-null   object 
 4   processor  1020 non-null   object 
 5   ram        1020 non-null   object 
 6   battery    1020 non-null   object 
 7   display    1020 non-null   object 
 8   camera     1019 non-null   object 
 9   card       1013 non-null   object 
 10  os         1003 non-null   object 
dtypes: float64(1), object(10)
memory usage: 87.8+ KB


### 2. Checking Numerical Summary Statistics

`describe()` helps inspect the distribution of numerical columns and can reveal suspicious values or unexpected ranges.

This is an assessment step rather than a cleaning step; the purpose is to understand the data before modifying it.


In [394]:
df.describe()

,rating
count,879.000000
mean,78.258248
std,7.402854
min,60.000000
25%,74.000000
50%,80.000000
75%,84.000000
max,89.000000


### 3. Checking for Duplicate Rows

Duplicate rows are checked using `duplicated().sum()`.

**Data Quality Dimension:** Uniqueness

Duplicate records can result in repeated observations and may distort later analysis.


In [395]:
df.duplicated().sum()

np.int64(0)

# 🧹 Cleaning the Smartphone Data

A copy of the original DataFrame is created before cleaning.

This keeps the raw dataset unchanged and provides a separate working DataFrame for transformations.


#Cleaning

In [396]:
df1=df.copy()

## 💰 Issue 1: Price Stored as Text

The `price` column contains the Indian rupee symbol and comma separators, so it is stored as text rather than as a numerical value.

**Data Quality Dimension:** Validity

**Issue Type:** Dirty data

**Problem:** Values such as `₹1,29,999` cannot be used directly for numerical analysis.

**Cleaning approach:** Remove `₹` and commas, then convert the result to an integer.


In [397]:
#price column
df1['price']=df1.price.str.replace('₹','').str.replace(',','').astype('int')

## 🔢 Preserving the Dataset's Original Row Identifier

The notebook resets the Pandas index and then adjusts the displayed `index` column to match the row numbering used in the source dataset.

This distinction is important:

- `df1.index` is the actual Pandas index.
- `df1['index']` is a column containing the source row identifier.

Keeping these concepts separate prevents index mismatches when identifying problematic rows.


In [398]:
df1=df1.reset_index()

In [399]:
df1['index']=df1['index']+2

In [400]:
df1

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,2,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,Motorola Moto Edge S30 Pro,34990,83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Android v12,No FM Radio
1016,1018,Honor X8 5G,14990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 480+, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,"6.5 inches, 720 x 1600 px Display with Water D...",48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,POCO X4 GT 5G (8GB RAM + 256GB),28990,85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Dimensity 8100, Octa Core, 2.85 GHz Processor","8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,Motorola Moto G91 5G,19990,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2400 px Display with Punch ...",108 MP + 8 MP + 2 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 1 TB",Android v12


## 🧩 Identifying Corrupted / Shifted Feature Rows

The dataset contains groups of row identifiers associated with suspected problems in different feature columns:

- processor
- RAM
- battery
- display
- camera

The sets below are used to locate rows where information appears to have shifted into the wrong column.

**Data Quality Dimension:** Accuracy / Validity

**Issue Type:** Dirty data

The next cells use set operations to identify rows affected by one or multiple column-level problems.


In [401]:
processor_rows = set((642,647,649,659,667,701,750,759,819,859,883,884,919,927,929,932,1002))
ram_rows = set((441,485,534,553,584,610,613,642,647,649,659,667,701,750,759,819,859,884,919,927,929,932,990,1002))
battery_rows = set((113,151,309,365,378,441,450,553,584,610,613,630,642,647,649,659,667,701,750,756,759,764,819,855,859,884,915,916,927,929,932,990,1002))
display_rows = set((378,441,450,553,584,610,613,630,642,647,649,659,667,701,750,759,764,819,859,884,915,916,927,929,932,990,1002))
camera_rows = set((100,113,151,157,161,238,273,308,309,323,324,365,367,378,394,441,450,484,506,534,553,571,572,575,584,610,613,615,630,642,647,649,659,667,684,687,705,711,723,728,750,756,759,764,792,819,846,854,855,858,883,884,896,915,916,927,929,932,945,956,990,995,1002,1016 ))

### 🔎 Finding Rows Affected by Any Identified Feature Issue

The union operator `|` combines the row sets.

This returns rows that appear in **at least one** of the issue lists.


In [402]:
df1[df1['index'].isin(processor_rows | ram_rows | battery_rows | display_rows | camera_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
155,157,Nokia 2780 Flip,4990,NaN,"Dual Sim, 3G, 4G, Wi-Fi","Snapdragon QM215, Quad Core, 1.3 GHz Processor","4 GB RAM, 512 MB inbuilt",1450 mAh Battery,"2.7 inches, 240 x 320 px Display",Dual Display,5 MP Rear Camera,"Memory Card Supported, upto 32 GB"
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
...,...,...,...,...,...,...,...,...,...,...,...,...
954,956,Vivo X Fold 5G (12GB RAM + 512GB),118990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 512 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
988,990,Nokia 5310 Dual Sim,3399,NaN,Dual Sim,"8 MB RAM, 16 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser
993,995,Huawei Mate X,169000,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Kirin 990, Octa Core, 2.86 GHz Processor","8 GB RAM, 512 GB inbuilt",4500 mAh Battery with 55W Fast Charging,"8 inches, 2200 x 2480 px Display",Foldable Display,48 MP Quad Rear Camera,"Memory Card (Hybrid), upto 256 GB"
1000,1002,XTouch F40 Flip,1999,NaN,Dual Sim,No 3G,No Wifi,"32 MB RAM, 32 MB inbuilt",800 mAh Battery,"1.77 inches, 240 x 320 px Display",Dual Display,1.3 MP Rear Camera


### 🔎 Finding Rows Affected by All Identified Feature Issues

The intersection operator `&` finds row identifiers common to all five issue sets.

This is useful for isolating records where several feature fields are affected simultaneously.


In [403]:
df1[df1['index'].isin(processor_rows & ram_rows & battery_rows & display_rows & camera_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
640,642,Nokia 105 Plus,1299,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",800 mAh Battery,"1.77 inches, 128 x 160 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,NaN
645,647,Nokia 2760 Flip,5490,NaN,"Dual Sim, 3G, 4G, Wi-Fi",1450 mAh Battery,"3.6 inches, 240 x 320 px Display",5 MP Rear & 5 MP Front Camera,"Memory Card Supported, upto 32 GB",Kaios v3.0,Bluetooth,NaN
647,649,Motorola Moto A10,1339,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",1750 mAh Battery,"1.8 inches, 160 x 128 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",NaN,NaN
657,659,Zanco Tiny T1,2799,NaN,Single Sim,"32 MB RAM, 32 MB inbuilt",200 mAh Battery,"0.49 inches, 64 x 32 px Display",No Rear Camera,No FM Radio,Bluetooth,NaN
665,667,itel it2163S,958,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",1200 mAh Battery,"1.8 inches, 160 x 128 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,NaN
748,750,Nokia 400 4G,3290,NaN,"Dual Sim, 4G, VoLTE, Wi-Fi",2000 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear & 0.3 MP Front Camera,"Memory Card Supported, upto 64 GB",Bluetooth,Browser,NaN
757,759,Karbonn KU3i,995,NaN,Dual Sim,"52 MB RAM, 32 MB inbuilt",1000 mAh Battery,"1.8 inches, 128 x 160 px Display",No Rear Camera,"Memory Card Supported, upto 16 GB",Bluetooth,NaN
817,819,itel Magic X,2239,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi",No 3G,T117,"48 MB RAM, 128 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",1.3 MP Rear Camera,"Memory Card Supported, upto 64 GB"
882,884,Nokia 5710 XpressAudio,4799,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
925,927,Nokia 3310 4G,3999,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","256 MB RAM, 512 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",2 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser


## 🗑️ Issue 2: Feature Phones / Records Outside the Analysis Scope

Rows with a price below ₹3,400 are removed from the working dataset.

**Cleaning decision:** Retain smartphones priced at or above ₹3,400.

This is a **data-filtering decision based on the scope of the analysis**, rather than a correction of an incorrect value.


In [404]:
df1=df1[df1['price']>=3400]#removed feature phones

### 🔎 Rechecking the Processor Issues After Filtering

After removing records below the chosen price threshold, the problematic-row sets are checked again.

Because filtering changes which rows remain in the working DataFrame, this step verifies which identified issues still require cleaning.


In [405]:
df1[df1['index'].isin(processor_rows & ram_rows & battery_rows & display_rows & camera_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
645,647,Nokia 2760 Flip,5490,NaN,"Dual Sim, 3G, 4G, Wi-Fi",1450 mAh Battery,"3.6 inches, 240 x 320 px Display",5 MP Rear & 5 MP Front Camera,"Memory Card Supported, upto 32 GB",Kaios v3.0,Bluetooth,NaN
882,884,Nokia 5710 XpressAudio,4799,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
925,927,Nokia 3310 4G,3999,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","256 MB RAM, 512 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",2 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser


### 🔎 Processor-Related Issues

The processor issue set is used to inspect the remaining affected records.

The row identifiers here refer to the source `index` column, so the filtering uses `df1['index']` rather than assuming that the current Pandas index has the same values.


In [406]:
df1[df1['index'].isin(processor_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
645,647,Nokia 2760 Flip,5490,NaN,"Dual Sim, 3G, 4G, Wi-Fi",1450 mAh Battery,"3.6 inches, 240 x 320 px Display",5 MP Rear & 5 MP Front Camera,"Memory Card Supported, upto 32 GB",Kaios v3.0,Bluetooth,NaN
857,859,LG Folder 2,11999,NaN,"Single Sim, 3G, 4G, Wi-Fi","1 GB RAM, 8 GB inbuilt",1470 mAh Battery,"2.8 inches, 240 x 320 px Display",2 MP Rear Camera,Memory Card Supported,Bluetooth,NaN
882,884,Nokia 5710 XpressAudio,4799,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
925,927,Nokia 3310 4G,3999,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","256 MB RAM, 512 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",2 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser


## 🗑️ Issue 3: Invalid Processor Records

Four processor-related records are removed because the affected rows are not suitable for reliable processor analysis.

**Data Quality Dimension:** Validity / Accuracy

**Issue Type:** Dirty data

The removal is based on the identified problematic source row identifiers.


In [407]:
df1.drop([645,857,882,925],inplace=True)

/tmp/ipykernel_2279/1676279141.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop([645,857,882,925],inplace=True)


In [408]:
df1[df1['index'].isin(processor_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os


## 🧠 Issue 4: RAM Information in the Wrong Position

The `ram` issue list is inspected to locate affected records.

Some records contain values that have shifted into adjacent fields. These cases are corrected later by repositioning the affected values.


In [409]:
df1[df1['index'].isin(ram_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15,No FM Radio
483,485,Huawei Mate 50 RS Porsche Design,239999,81.0,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi, NFC, IR Blaster","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor",512 GB inbuilt,4700 mAh Battery with 66W Fast Charging,"6.74 inches, 1212 x 2616 px, 120 Hz Display",50 MP + 48 MP + 13 MP Triple Rear & 13 MP Fron...,"Memory Card (Hybrid), upto 256 GB",Hongmeng OS v3.0
582,584,Nokia 8210 4G,3749,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.8 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"


## 🔋 Issue 5: Battery Information in the Wrong Position

The battery issue list identifies records where the battery information is not aligned with the expected columns.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

The affected rows are inspected before values are shifted back into their intended columns.


In [410]:
df1[df1['index'].isin(battery_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
376,378,Nokia 2660 Flip,4649,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.8 inches, 240 x 320 px Display",Dual Display,0.3 MP Rear Camera
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15,No FM Radio
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt","6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15,No FM Radio
582,584,Nokia 8210 4G,3749,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.8 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0,No FM Radio
754,756,Apple iPod Touch (7th Gen),18900,NaN,Wi-Fi,32 GB inbuilt,"4 inches, 640 x 1136 px Display",8 MP Rear & 1.2 MP Front Camera,iOS v12,No FM Radio,Bluetooth,Browser


### 🗑️ Removing Invalid Battery Records

Two identified rows are removed because their battery-related information cannot be reliably corrected using the available structure.


In [411]:
df1.drop([376,754],inplace=True)

/tmp/ipykernel_2279/2579486625.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop([376,754],inplace=True)


### 🗑️ Removing Another Invalid Battery Record

A further problematic row is removed after inspection of the battery-related records.


In [412]:
df1.drop([582],inplace=True)

/tmp/ipykernel_2279/2080557543.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop([582],inplace=True)


### 🛠️ Correcting Shifted Battery Data

The affected battery rows are isolated into a temporary DataFrame.

The next operations shift the feature values so that information is returned to the intended columns.


In [413]:
temp_df=df1[df1['index'].isin(battery_rows)]

In [414]:
x=temp_df.iloc[:,7:].shift(1,axis=1).values

In [415]:
x_index=temp_df.index

In [416]:
df1.loc[temp_df.index,temp_df.columns[7:]]=x

In [417]:
df1[df1['index'].isin(battery_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt",None,"6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
762,764,Apple iPhone SE 4,49990,60.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"6.1 inches, 750 x 1580 px Display",12 MP Rear & 10.8 MP Front Camera,Memory Card Not Supported,iOS v16
853,855,Apple iPhone 12 Pro (256GB),119900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 256 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
913,915,Apple iPhone 12 Mini (256GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


## 🖥️ Issue 6: Display Information in the Wrong Column

The display issue set is inspected.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

The affected rows are investigated separately because values may have shifted between adjacent feature columns.


In [418]:
df1[df1['index'].isin(display_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt",None,"6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
762,764,Apple iPhone SE 4,49990,60.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"6.1 inches, 750 x 1580 px Display",12 MP Rear & 10.8 MP Front Camera,Memory Card Not Supported,iOS v16
913,915,Apple iPhone 12 Mini (256GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
914,916,Apple iPhone 12 (256GB),67999,76.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


## 📷 Issue 7: Camera Information in the Wrong Column

The camera issue set identifies rows requiring further inspection.

Some records contain camera-related information in another feature field. These rows are corrected by moving the appropriate values back to the `camera` column.


In [419]:
df1[df1['index'].isin(camera_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
155,157,Nokia 2780 Flip,4990,NaN,"Dual Sim, 3G, 4G, Wi-Fi","Snapdragon QM215, Quad Core, 1.3 GHz Processor","4 GB RAM, 512 MB inbuilt",1450 mAh Battery,"2.7 inches, 240 x 320 px Display",Dual Display,5 MP Rear Camera,"Memory Card Supported, upto 32 GB"
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
236,238,Xiaomi Mix Fold 2 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 8+ Gen1 , Octa Core, 3.2 GHz Proce...","12 GB RAM, 256 GB inbuilt",4500 mAh Battery with 67W Fast Charging,"8.02 inches, 1914 x 2160 px, 120 Hz Display wi...","Foldable Display, Dual Display",50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,Android v12
271,273,Nokia 2720 V Flip,6199,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","Snapdragon 205 , Dual Core, 1.1 GHz Processor","512 MB RAM, 4 GB inbuilt",1500 mAh Battery,"2.8 inches, 240 x 320 px Display",Dual Display,2 MP Rear Camera,Memory Card Supported
306,308,Samsung Galaxy Z Flip 3,69999,84.0,"Single Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3300 mAh Battery with 15W Fast Charging,"6.7 inches, 1080 x 2640 px, 120 Hz Display wit...","Foldable Display, Dual Display",12 MP + 12 MP Dual Rear & 10 MP Front Camera,Memory Card Not Supported
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
321,323,Samsung Galaxy Z Fold 4,154998,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4400 mAh Battery with 25W Fast Charging,"7.6 inches, 1812 x 2176 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,Android v12


### 🗑️ Removing Invalid Camera Records

Two identified rows are removed because their camera-related information cannot be reliably recovered from the available data.


In [420]:
df1.drop([155,271],inplace=True)

/tmp/ipykernel_2279/1298457551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop([155,271],inplace=True)


In [421]:
df1[df1['index'].isin(camera_rows)]


,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
236,238,Xiaomi Mix Fold 2 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 8+ Gen1 , Octa Core, 3.2 GHz Proce...","12 GB RAM, 256 GB inbuilt",4500 mAh Battery with 67W Fast Charging,"8.02 inches, 1914 x 2160 px, 120 Hz Display wi...","Foldable Display, Dual Display",50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,Android v12
306,308,Samsung Galaxy Z Flip 3,69999,84.0,"Single Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3300 mAh Battery with 15W Fast Charging,"6.7 inches, 1080 x 2640 px, 120 Hz Display wit...","Foldable Display, Dual Display",12 MP + 12 MP Dual Rear & 10 MP Front Camera,Memory Card Not Supported
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
321,323,Samsung Galaxy Z Fold 4,154998,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4400 mAh Battery with 25W Fast Charging,"7.6 inches, 1812 x 2176 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,Android v12
322,324,Royole FlexPai 2,109999,87.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 865, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",4450 mAh Battery,"7.8 inches, 1440 x 1920 px Display","Foldable Display, Dual Display",64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 256 GB"
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


### 🛠️ Inspecting Camera Records

The affected camera records are isolated so that their values can be examined before correction.


In [422]:
temp_df=df1[df1['index'].isin(camera_rows)]

### 🔎 Detecting Camera Rows Without the Expected `MP` Pattern

Camera values are expected to contain megapixel information such as `50 MP`.

Rows that do not contain the expected pattern are isolated for further inspection.


In [423]:
temp_df=temp_df[~temp_df['camera'].str.contains('MP')]

### 🛠️ Restoring Camera Values

For the identified rows, the value from the `card` column is moved into the `camera` column.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

This indicates that values had been shifted into the wrong column in the original data.


In [424]:
df1.loc[temp_df.index,'camera']=temp_df['card'].values

In [425]:
df1[df1['index'].isin(camera_rows)]

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",50 MP Quad Rear & 16 MP Front Camera,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
236,238,Xiaomi Mix Fold 2 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 8+ Gen1 , Octa Core, 3.2 GHz Proce...","12 GB RAM, 256 GB inbuilt",4500 mAh Battery with 67W Fast Charging,"8.02 inches, 1914 x 2160 px, 120 Hz Display wi...",50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,Android v12
306,308,Samsung Galaxy Z Flip 3,69999,84.0,"Single Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3300 mAh Battery with 15W Fast Charging,"6.7 inches, 1080 x 2640 px, 120 Hz Display wit...",12 MP + 12 MP Dual Rear & 10 MP Front Camera,12 MP + 12 MP Dual Rear & 10 MP Front Camera,Memory Card Not Supported
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
321,323,Samsung Galaxy Z Fold 4,154998,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4400 mAh Battery with 25W Fast Charging,"7.6 inches, 1812 x 2176 px, 120 Hz Display wit...",50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,Android v12
322,324,Royole FlexPai 2,109999,87.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 865, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",4450 mAh Battery,"7.8 inches, 1440 x 1920 px Display",64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 256 GB"
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


## 💾 Issue 8: Card and Other Feature Values Are Misplaced

The value counts of the `card` column reveal suspicious categories such as operating-system values appearing in the card column.

This is a strong sign that some records have **shifted values**.

**Data Quality Dimension:** Accuracy / Consistency

**Issue Type:** Dirty data


In [426]:
df['card'].value_counts()

,count
card,
"Memory Card Supported, upto 1 TB",171
Memory Card Not Supported,112
Android v12,107
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
...,...
"Memory Card Supported, upto 48 GB",1
"Memory Card (Hybrid), upto 2 TB",1
HarmonyOS,1


### 🔎 Locating Rows Where OS Information Appears in `card`

Rows containing `MP` in the card field are isolated for inspection because this pattern suggests that camera-related information may have shifted into the card column.


In [427]:
temp_df=df1[df1['card'].str.contains('MP')]

### 🛠️ Correcting the Card Field

The identified values are replaced with `Memory Card Not Supported`.

This standardizes the card information for those records.


In [428]:
df1.loc[temp_df.index,'card']='Memory Card Not Supported'

### 📊 Rechecking Card Value Counts

The value counts are recalculated to verify the effect of the correction.

Comparing counts before and after a cleaning operation is a useful validation step.


In [429]:
df1['card'].value_counts()

,count
card,
"Memory Card Supported, upto 1 TB",171
Memory Card Not Supported,149
Android v12,107
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
Memory Card Supported,89
"Memory Card Supported, upto 256 GB",87
Android v13,46
Android v11,41


### 🔎 Finding Rows Where Card Information Is Missing or Misplaced

Rows that do not contain the expected memory-card wording are isolated for further inspection.


In [430]:
temp_df=df1[~df1['card'].str.contains('Memory Card')]

### 🛠️ Restoring OS Values

For these records, the value currently stored in `card` is moved into the `os` column.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

This corrects values that were shifted one column to the left or right.


In [431]:
df1.loc[temp_df.index,'os']=temp_df['card'].values

### 🛠️ Standardizing the Card Field

After restoring the OS value, the card field is standardized to `Memory Card Not Supported`.


In [432]:
df1.loc[temp_df.index,'card']='Memory Card Not Supported'

### 📊 Validating the Card Cleaning

The value counts are checked again to confirm how the distribution changed after the corrections.

This provides a simple validation that the cleaning operations affected the intended category.


In [433]:
df1['card'].value_counts()

,count
card,
Memory Card Not Supported,362
"Memory Card Supported, upto 1 TB",171
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
Memory Card Supported,89
"Memory Card Supported, upto 256 GB",87
Memory Card (Hybrid),30
"Memory Card (Hybrid), upto 256 GB",13
"Memory Card (Hybrid), upto 512 GB",11


### 📊 Checking Operating-System Values

The `os` value counts are inspected after the column-shift corrections.

This helps identify remaining values that do not belong in the operating-system field.


In [434]:
df1['os'].value_counts()

,count
os,
Android v12,394
Android v11,274
Android v13,91
Android v10,69
Android v9.0 (Pie),29
Android v10.0,23
iOS v16,15
iOS v15,12
Android v8.1 (Oreo),10


## 🔎 Issue 9: Memory-Card Values Appearing in the OS Column

Rows where the `os` field contains memory-card information are isolated.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data

These values belong in the `card` column rather than the `os` column.


In [435]:
temp_df=df1[df1['os'].str.contains('Memory Card')]

### 🔎 Identifying Rows Where Card and OS Contain Different Values

Rows where the OS contains memory-card information but differs from the card value are isolated.

This prevents blindly overwriting records where the two fields may already contain the same information.


In [436]:
temp_df=temp_df[~(temp_df['card']==temp_df['os'])]

In [437]:
temp_df

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
322,324,Royole FlexPai 2,109999,87.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 865, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",4450 mAh Battery,"7.8 inches, 1440 x 1920 px Display",64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,Memory Card Not Supported,"Memory Card Supported, upto 256 GB"
570,572,LG Wing 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 765G , Octa Core, 2.4 GHz Processor","8 GB RAM, 128 GB inbuilt",4000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2460 px Display",64 MP + 13 MP + 12 MP Triple Rear & 32 MP Fron...,Memory Card Not Supported,"Memory Card (Hybrid), upto 2 TB"
613,615,Oukitel WP21,22990,82.0,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","Helio G99, Octa Core, 2.2 GHz Processor","12 GB RAM, 256 GB inbuilt",9800 mAh Battery with 66W Fast Charging,"6.78 inches, 1080 x 2400 px, 120 Hz Display",64 MP + 20 MP + 2 MP Triple Rear & Main Front ...,Memory Card Not Supported,Memory Card (Hybrid)
682,684,LG V60 ThinQ,79990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 865, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2460 px Display with Punch ...",64 MP + 13 MP + 0.3 MP Triple Rear & 10 MP Fro...,Memory Card Not Supported,"Memory Card (Hybrid), upto 2 TB"
726,728,Huawei Mate Xs 2,162990,89.0,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi, NFC, IR Blaster","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 512 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"7.8 inches, 2200 x 2480 px, 120 Hz Display wit...",50 MP + 13 MP + 8 MP Triple Rear & 10.7 MP Fro...,Memory Card Not Supported,"Memory Card (Hybrid), upto 256 GB"
844,846,CAT S22 Flip,14999,NaN,"Single Sim, 3G, 4G, VoLTE, Wi-Fi","Qualcomm 215, Quad Core, 1.3 GHz Processor","2 GB RAM, 16 GB inbuilt",2000 mAh Battery,"4 inches, 480 x 640 px Display",5 MP Rear & 2 MP Front Camera,Memory Card Not Supported,"Memory Card Supported, upto 128 GB"
894,896,Royole FlexPai 3 5G,149999,87.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3360 mAh Battery,"7.2 inches, 1440 x 1920 px Display",64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,Memory Card Not Supported,"Memory Card Supported, upto 256 GB"
993,995,Huawei Mate X,169000,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Kirin 990, Octa Core, 2.86 GHz Processor","8 GB RAM, 512 GB inbuilt",4500 mAh Battery with 55W Fast Charging,"8 inches, 2200 x 2480 px Display",48 MP Quad Rear Camera,Memory Card Not Supported,"Memory Card (Hybrid), upto 256 GB"


### 🛠️ Moving Misplaced Memory-Card Information

The memory-card value is restored to the `card` column.

This corrects the location of the information without discarding the original value.


In [438]:
df1.loc[temp_df.index,'card']=temp_df['os'].values

### 🔎 Rechecking Remaining Misplaced OS Values

The OS column is checked again for values containing `Memory Card`.

This is a validation step to find any remaining misplaced values.


In [439]:
temp_df=df1[df1['os'].str.contains('Memory Card')]

### 🧹 Removing Invalid OS Values

Remaining memory-card text is replaced with missing values (`NaN`) in the OS column because it does not represent an operating system.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [440]:
df1.loc[temp_df.index,'os']=np.nan

### 🔎 Finding Another Invalid OS Value

`Bluetooth` is identified as another value that does not belong in the operating-system field.


In [441]:
temp_df=df1[df1['os']=='Bluetooth']

### 🧹 Removing the Invalid OS Value

The invalid `Bluetooth` value is replaced with `NaN`.

This preserves the record while correctly representing that the OS information is unavailable.


In [442]:
df1.loc[temp_df.index,'os']=np.nan

### 📊 Validating the OS Cleaning

The OS value counts are recalculated after removing values that do not belong in the column.


In [443]:
df1['os'].value_counts()

,count
os,
Android v12,394
Android v11,274
Android v13,91
Android v10,69
Android v9.0 (Pie),29
Android v10.0,23
iOS v16,15
iOS v15,12
Android v8.1 (Oreo),10


## 📐 Checking the Cleaned Dataset Shape

The number of rows and columns is checked after the major cleaning operations.

This provides a quick validation of how much the dataset changed.


In [444]:
df1.shape

(982, 12)

## 🏷️ Feature Engineering: Extracting Brand Name

The brand name is extracted from the first word of the smartphone model.

This creates a new structured variable that is easier to use for grouping and analysis.


In [445]:
brand_names=df1.model.str.split(' ').str.get(0)

### ➕ Adding the `brand_name` Column

The extracted brand is inserted into the DataFrame as a separate column.


In [446]:
df1.insert(1,'brand_name',brand_names)

### 🔤 Standardizing Brand Names

Brand names are converted to lowercase to maintain a consistent representation.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data

Standardized text reduces problems caused by capitalization differences.


In [447]:
df1['brand_name']=df1.brand_name.str.lower()

/tmp/ipykernel_2279/1445710651.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['brand_name']=df1.brand_name.str.lower()


## 📶 Feature Engineering: SIM Capabilities

The `sim` field contains several capabilities together.

Three Boolean features are extracted:

- `has_5g`
- `has_nfc`
- `has_ir_blaster`

This converts embedded information into separate analysis-ready variables.

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data


In [448]:
has_5g = df1['sim'].str.contains('5G')
has_nfc = df1['sim'].str.contains('NFC')
has_ir_blaster = df1['sim'].str.contains('IR Blaster')

### ➕ Adding SIM Capability Features

The Boolean variables are inserted as separate columns.

Separating these attributes makes filtering and aggregation easier.


In [449]:
df1.insert(6,'has_5g',has_5g)
df1.insert(7,'has_nfc',has_nfc)
df1.insert(8,'has_ir_blaster',has_ir_blaster)

## ⚙️ Feature Engineering: Processor Information

The original `processor` column contains multiple pieces of information together.

The notebook separates it into:

- processor name
- number of cores
- processor speed

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data

Multiple variables stored in one column are separated into individual fields.


In [450]:
processor_name=df1.processor.str.split(',').str.get(0)


In [451]:
num_cores = df1['processor'].str.split(',').str.get(1)

In [452]:
processor_speed = df1['processor'].str.split(',').str.get(2)

In [453]:
df1.insert(10,'processor_name',processor_name)
df1.insert(11,'num_cores',num_cores)
df1.insert(12,'processor_speed',processor_speed)

### 🧹 Standardizing Processor Names

Whitespace is removed from processor names to make their text representation consistent.


In [454]:
df1['processor_name'] = df1['processor_name'].str.strip()

/tmp/ipykernel_2279/1632474951.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['processor_name'] = df1['processor_name'].str.strip()


### 🔄 Correcting Shifted Processor Components

Rows containing `Core` in the processor name are inspected because the processor components are shifted relative to the expected structure.

The next cell shifts the values back into their intended fields.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [455]:
temp_df=df1[df1['processor_name'].str.contains('Core')][['processor_name', 'num_cores',	'processor_speed']].shift(1,axis=1)

In [456]:
df1.loc[temp_df.index,['processor_name', 'num_cores',	'processor_speed']]=temp_df.values

### 🔎 Inspecting a Specific Processor Record

A specific row is inspected after the processor-column correction to verify the values before applying a targeted fix.


In [457]:
df1.loc[856]

,856
index,858
brand_name,samsung
model,Samsung Galaxy A01 Core
price,4999
rating,NaN
sim,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi"
has_5g,False
has_nfc,False
has_ir_blaster,False
processor,"(28 nm), Quad Core, 1.5 GHz Processor"


### 🛠️ Correcting the Processor Name

The processor name for the identified record is corrected to `Mediatek MT6739`.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [458]:
df1.loc[856,'processor_name'] = 'Mediatek MT6739'

### 🏷️ Extracting Processor Brand

The processor brand is extracted from the processor name and stored as a separate variable.

This makes processor-level analysis easier.


In [459]:
processor_brand = df1['processor_name'].str.split(' ').str.get(0).str.lower()
df1.insert(11,'processor_brand',processor_brand)

### 🧹 Cleaning the Number-of-Cores Field

Leading and trailing whitespace is removed from the `num_cores` values.


In [460]:
df1['num_cores'] = df1['num_cores'].str.strip()

/tmp/ipykernel_2279/2483753889.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['num_cores'] = df1['num_cores'].str.strip()


### 🔤 Standardizing Core Descriptions

Different descriptions such as `Octa Core Processor` and `Octa Core` are standardized to the same representation.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data


In [461]:
df1['num_cores'] = df1['num_cores'].str.replace('Octa Core Processor','Octa Core').str.replace('Hexa Core Processor','Hexa Core')

/tmp/ipykernel_2279/3285793486.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['num_cores'] = df1['num_cores'].str.replace('Octa Core Processor','Octa Core').str.replace('Hexa Core Processor','Hexa Core')


### 🔢 Converting Processor Speed to a Numeric Value

Processor speed is cleaned by removing extra whitespace and extracting the numeric component before converting it to a float.

This makes the field suitable for numerical analysis.

**Data Quality Dimension:** Validity


In [462]:
df1['processor_speed'] = df1['processor_speed'].str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0).astype(float)

/tmp/ipykernel_2279/825004774.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['processor_speed'] = df1['processor_speed'].str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0).astype(float)


## 💾 Feature Engineering: RAM and Internal Memory

The original `ram` field contains multiple variables.

The notebook extracts:

- `ram_capacity`
- `internal_memory`

This converts a combined text field into separate numeric features.

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data


In [463]:
ram_capacity=df1['ram'].str.strip().str.split(',').str.get(0).str.findall(r'\b(\d+)\b').str.get(0)

In [464]:
df1.insert(16,'ram_capacity',ram_capacity)

In [465]:
internal_memory = df1['ram'].str.strip().str.split(',').str.get(1).str.strip().str.findall(r'\b(\d+)\b').str.get(0)

In [466]:
df1.insert(17,'internal_memory',internal_memory)

### 🔢 Converting RAM Capacity to Numeric

`ram_capacity` is converted to a floating-point number so that it can be used in numerical analysis.


In [467]:
df1['ram_capacity'] = df1['ram_capacity'].astype(float)

/tmp/ipykernel_2279/2073605403.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['ram_capacity'] = df1['ram_capacity'].astype(float)


### 🗑️ Removing Invalid RAM Records

Two records are removed after identifying RAM values that cannot be reliably interpreted using the available data.


In [468]:
df1.drop([486,627],inplace=True)

/tmp/ipykernel_2279/2098217776.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop([486,627],inplace=True)


### 🛠️ Correcting a RAM / Internal Memory Record

The identified record is manually corrected using the intended RAM capacity and internal-memory values.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [469]:
df1.loc[[483], ['ram_capacity','internal_memory']] = [12.0,'512']

### 🔢 Converting Internal Memory to Numeric

The `internal_memory` field is converted to a numeric type.


In [470]:
df1['internal_memory'] = df1['internal_memory'].astype(float)

/tmp/ipykernel_2279/3988935729.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['internal_memory'] = df1['internal_memory'].astype(float)


### 🔎 Identifying Suspicious Internal-Memory Values

Records with `internal_memory == 1` are isolated for inspection.

The next step treats these values as a likely unit-related representation rather than a literal 1 GB value.


In [471]:
temp_df = df1[df1['internal_memory'] == 1]

### 🛠️ Standardizing Internal Memory

The identified value is converted from `1` to `1024`, representing the same storage quantity in a consistent unit.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data


In [472]:
df1.loc[temp_df.index,'internal_memory'] = 1024

## 🔋 Feature Engineering: Battery Capacity

Battery information is extracted from the original `battery` field.

The numeric battery capacity is separated from the fast-charging information.


In [473]:
battery_capacity = df1['battery'].str.strip().str.split('with').str.get(0).str.strip().str.findall(r'\b(\d+)\b').str.get(0).astype(float)

In [474]:
df1.insert(16,'battery_capacity',battery_capacity)

### ⚡ Extracting Fast-Charging Information

Fast-charging information is extracted from the battery description.

Because the source field may contain different text patterns, a custom extraction function is used in the following cells.


In [475]:
fast_charging = df1['battery'].str.strip().str.split('with').str.get(1).str.strip().str.findall(r'\d{2,3}')

In [476]:
df1.insert(17,'fast_charging',fast_charging)

### 🧩 Handling Different Fast-Charging Formats

The custom function handles different extraction results:

- one detected value → return that value
- multiple values → use `0`
- no list → use `-1`

This creates a consistent numeric representation for the new field.


In [477]:
def fast_charging_extractor(item):

  if type(item) == list:
    if len(item) == 1:
      return item[0]
    else:
      return 0
  else:
    return -1

In [478]:
df1['fast_charging'] = df1['fast_charging'].apply(fast_charging_extractor).astype(int)

/tmp/ipykernel_2279/3894870647.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['fast_charging'] = df1['fast_charging'].apply(fast_charging_extractor).astype(int)


## 🖥️ Feature Engineering: Display Information

The original `display` field contains several variables together.

The notebook extracts:

- screen size
- resolution
- refresh rate

**Data Quality Dimension:** Tidiness

**Issue Type:** Messy data


In [479]:
screen_size = df1['display'].str.strip().str.split(',').str.get(0).str.strip().str.split(' ').str.get(0).astype(float)

In [480]:
df1.insert(21,'screen_size',screen_size)

In [481]:
resolution = df1['display'].str.strip().str.split(',').str.get(1).str.strip().str.split('px').str.get(0)

In [482]:
df1.insert(22,'resolution',resolution)

### 🔄 Standardizing Missing Refresh Rates

Where a refresh rate is not available, the notebook assigns `60` as the default value before converting the field to integer type.

This is a cleaning assumption and should be interpreted as a chosen default rather than a directly observed value.


In [483]:
refresh_rate = df1['display'].str.strip().str.split(',').str.get(2).str.strip().str.findall(r'\d{2,3}').str.get(0).apply(lambda x: 60 if pd.isna(x) else x).astype(int)

In [484]:
df1.insert(22,'refresh_rate',refresh_rate)

In [485]:
df1

,index,brand_name,model,price,rating,sim,has_5g,has_nfc,has_ir_blaster,processor,processor_name,processor_brand,num_cores,processor_speed,ram,battery,battery_capacity,fast_charging,ram_capacity,internal_memory,display,screen_size,refresh_rate,resolution,camera,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",True,True,False,"Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor",Snapdragon 8 Gen2,snapdragon,Octa Core,3.20,"12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,5000.0,100,12.0,256.0,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",6.70,120,1440 x 3216,50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Snapdragon 695, Octa Core, 2.2 GHz Processor",Snapdragon 695,snapdragon,Octa Core,2.20,"6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,5000.0,33,6.0,128.0,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",6.59,120,1080 x 2412,64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,samsung,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Exynos 1330, Octa Core, 2.4 GHz Processor",Exynos 1330,exynos,Octa Core,2.40,"4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,5000.0,15,4.0,64.0,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",6.60,90,1080 x 2408,50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,motorola,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Snapdragon 695, Octa Core, 2.2 GHz Processor",Snapdragon 695,snapdragon,Octa Core,2.20,"6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,5000.0,0,6.0,128.0,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",6.55,120,1080 x 2400,50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,realme,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Dimensity 1080, Octa Core, 2.6 GHz Processor",Dimensity 1080,dimensity,Octa Core,2.60,"6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,5000.0,67,6.0,128.0,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",6.70,120,1080 x 2412,108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,motorola,Motorola Moto Edge S30 Pro,34990,83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Snapdragon 8 Gen1, Octa Core, 3 GHz Processor",Snapdragon 8 Gen1,snapdragon,Octa Core,3.00,"8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,5000.0,68,8.0,128.0,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",6.67,120,1080 x 2460,64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1016,1018,honor,Honor X8 5G,14990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",True,False,False,"Snapdragon 480+, Octa Core, 2.2 GHz Processor",Snapdragon 480+,snapdragon,Octa Core,2.20,"6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,5000.0,22,6.0,128.0,"6.5 inches, 720 x 1600 px Display with Water D...",6.50,60,720 x 1600,48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,poco,POCO X4 GT 5G (8GB RAM + 256GB),28990,85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...",True,True,True,"Dimensity 8100, Octa Core, 2.85 GHz Processor",Dimensity 8100,dimensity,Octa Core,2.85,"8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,5080.0,67,8.0,256.0,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",6.60,144,1080 x 2460,64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,motorola,Motorola Moto G91 5G,19990,80.0,"Dual Sim, 3G, 4G, 5G, VoLT

## 📷 Feature Engineering: Camera Information

The original `camera` field contains descriptions of rear and front cameras.

A helper function converts descriptions such as:

- Quad → 4
- Triple → 3
- Dual → 2
- Missing → Missing
- other single-camera descriptions → 1

This produces structured camera-count variables.


In [486]:
def camera_extractor(text):

  if 'Quad' in text:
    return '4'
  elif 'Triple' in text:
    return '3'
  elif 'Dual' in text:
    return '2'
  elif 'Missing' in text:
    return 'Missing'
  else:
    return '1'

### 📸 Extracting Number of Rear Cameras

The rear-camera description is separated from the combined camera field and converted into a numeric camera count.


In [487]:
num_rear_cameras = df1['camera'].str.strip().str.split('&').str.get(0).apply(camera_extractor)

In [488]:
df1.insert(25,'num_rear_cameras',num_rear_cameras)

### 🤳 Extracting Number of Front Cameras

The front-camera description is extracted from the second part of the camera field.

Missing front-camera information is represented as `Missing` before applying the camera-count extractor.


In [489]:
num_front_cameras = df1['camera'].str.strip().str.split('&').str.get(1).str.strip().fillna('Missing').apply(camera_extractor)

In [490]:
df1.insert(26,'num_front_cameras',num_front_cameras)

In [491]:
pd.set_option('display.max_columns',None)

## 🔎 Investigating an Unusual Camera Value

A specific camera value is checked before applying a targeted correction.

This is an example of validating a suspicious record rather than changing values without inspection.


In [492]:
df1.loc[69,'camera'] == '50 MP'

False

### 🔎 Identifying Incorrect Camera Descriptions

The value `Foldable Display, Dual Display` is isolated because it does not describe a camera configuration and therefore does not belong in the `camera` column.

**Data Quality Dimension:** Accuracy

**Issue Type:** Dirty data


In [493]:
temp_df = df1[df1['camera'] == 'Foldable Display, Dual Display']

In [494]:
temp_df

,index,brand_name,model,price,rating,sim,has_5g,has_nfc,has_ir_blaster,processor,processor_name,processor_brand,num_cores,processor_speed,ram,battery,battery_capacity,fast_charging,ram_capacity,internal_memory,display,screen_size,refresh_rate,resolution,camera,num_rear_cameras,num_front_cameras,card,os
69,71,oppo,Oppo Find N Fold,99990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",True,True,False,"Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor",Snapdragon 8+ Gen1,snapdragon,Octa Core,3.2,"8 GB RAM, 256 GB inbuilt",5000 mAh Battery with 67W Fast Charging,5000.0,67,8.0,256.0,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...",7.1,120,1792 x 1920,"Foldable Display, Dual Display",2,Missing,Memory Card Not Supported,Android v12


### 🛠️ Correcting the Camera Value

The incorrectly placed camera value is corrected to `50 MP`.

This restores the intended camera information for the affected record.


In [495]:
df1.loc[temp_df.index, 'camera'] = '50 MP'

### 📷 Extracting Primary Rear Camera

The primary rear-camera megapixel value is extracted into a separate column.

This makes the camera specification easier to use for analysis.


In [496]:
df1['primary_camera_rear'] = df1['camera'].str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)

/tmp/ipykernel_2279/3974806889.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['primary_camera_rear'] = df1['camera'].str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)


### 🤳 Extracting Primary Front Camera

The primary front-camera megapixel value is extracted into a separate column.


In [497]:
df1['primary_camera_front'] = df1['camera'].str.split('&').str.get(1).str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)

/tmp/ipykernel_2279/1434923224.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['primary_camera_front'] = df1['camera'].str.split('&').str.get(1).str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)


## 💾 Issue 10: Ambiguous Hybrid Memory Card Values

Rows with `Memory Card (Hybrid)` are isolated for inspection.

The notebook standardizes these ambiguous entries to `Not Specified`.


In [499]:
temp_df=df1[df1['card'] == 'Memory Card (Hybrid)']

### 🛠️ Standardizing Hybrid Memory Card Information

`Memory Card (Hybrid)` is replaced with `Not Specified`.

**Data Quality Dimension:** Consistency

**Issue Type:** Dirty data

The goal is to use a consistent category for cases where the extended-memory information is not explicitly specified.


In [500]:
df1.loc[temp_df.index, 'card'] = 'Not Specified'

## 📊 Creating the `extended_memory` Feature

A final structured feature is created from the `card` column.

The transformation distinguishes unsupported / unspecified memory-card information from supported capacity values and extracts the capacity where available.

This is another example of turning a complex text field into an analysis-ready variable.


In [501]:
df1['extended_memory'] = df1['card'].apply(lambda x:'0' if 'Not' in x else x.split('upto')).str.get(-1).str.strip().str.replace('Memory Card Supported','Not Specified')

/tmp/ipykernel_2279/1471866915.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['extended_memory'] = df1['card'].apply(lambda x:'0' if 'Not' in x else x.split('upto')).str.get(-1).str.strip().str.replace('Memory Card Supported','Not Specified')


# 📦 Preparing the Final Analysis Dataset

The original complex text columns are removed after the useful information has been extracted into structured columns.

The resulting `export_df` is intended to be the cleaner, analysis-ready version of the smartphone dataset.


In [507]:
export_df = df1.drop(columns=['index','sim','processor','ram','battery','display','camera','card'])

## 💾 Exporting the Cleaned Dataset

The cleaned DataFrame is exported as:

`smartphone_cleaned_v2.csv`

The CSV is written without the Pandas index because the final dataset already contains the required analytical columns.


In [508]:
export_df.to_csv('smartphone_cleaned_v2.csv',index=False)

# 📌 Final Summary

This notebook follows a complete smartphone data-cleaning and preprocessing workflow.

### 🔎 Assessment
- Inspected the raw dataset
- Checked missing values and data types
- Checked duplicate records
- Examined numerical summaries

### 🧹 Data Cleaning
- Cleaned currency-formatted prices
- Identified and removed invalid records
- Corrected values shifted into the wrong columns
- Standardized text values
- Corrected processor, RAM, battery, display, camera, card, and OS information
- Handled ambiguous and missing values

### 🧩 Data Tidiness & Feature Engineering
- Extracted smartphone brands
- Split SIM capabilities into Boolean features
- Split processor information into separate variables
- Extracted RAM and internal memory
- Extracted battery capacity and fast-charging information
- Extracted display size, resolution, and refresh rate
- Extracted rear and front camera information
- Created extended-memory information

### 📚 Data Quality Concepts Demonstrated

**Completeness · Validity · Accuracy · Consistency · Uniqueness · Tidiness**

The final result is exported as `smartphone_cleaned_v2.csv` and is ready for further exploratory data analysis.
